# 🤖 Roboshelf AI — Fázis 2: G1 Retail Navigáció (Kaggle GPU)

**A G1 humanoid megtanul járni a retail boltban a raktárig.**

- MuJoCo + Unitree G1 MJCF + egyedi retail bolt XML
- Stable-Baselines3 PPO, párhuzamos környezetekkel
- Reward: előre haladás + egyensúly + contact pattern (gait) + célba érés bonus
- Kaggle T4 GPU (~2-4 óra, `teljes` szint)

⚡ **Accelerator: GPU T4 x2** (Settings → Accelerator)

In [ ]:
# === 1. GPU + PYTHON ELLENŐRZÉS ===
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                         '--format=csv,noheader'], capture_output=True, text=True)
print('🖥️  GPU:')
for line in result.stdout.strip().splitlines():
    print(f'   {line}')

import sys
print(f'\n🐍 Python: {sys.version.split()[0]}')

import torch
print(f'🔥 PyTorch: {torch.__version__}')
print(f'   CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# === 2. CSOMAGOK TELEPÍTÉSE ===
import subprocess, sys, os

# pip install subprocess-zel (\! magic nem mindig működik papermill alatt)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'mujoco>=3.6.0', 'gymnasium', 'stable-baselines3>=2.7.0', 'tensorboard'],
    check=True)
print('✅ Csomagok telepítve')

# mujoco_menagerie klónozása /tmp/-be
MENAGERIE_PATH = '/tmp/mujoco_menagerie'
if not os.path.exists(MENAGERIE_PATH):
    subprocess.run(['git', 'clone', '--depth', '1',
        'https://github.com/google-deepmind/mujoco_menagerie.git',
        MENAGERIE_PATH], check=True)
    print('✅ mujoco_menagerie klónozva → /tmp/')
else:
    print('✅ mujoco_menagerie már megvan (/tmp/)')

import mujoco, gymnasium, stable_baselines3
print(f'✅ MuJoCo:    {mujoco.__version__}')
print(f'✅ Gymnasium: {gymnasium.__version__}')
print(f'✅ SB3:       {stable_baselines3.__version__}')

# G1 XML gyors ellenőrzés
from pathlib import Path
g1_path = Path(MENAGERIE_PATH) / 'unitree_g1'
assert (g1_path / 'g1.xml').exists(), 'g1.xml nem található\!'
model_test = mujoco.MjModel.from_xml_path(str(g1_path / 'g1.xml'))
print(f'✅ G1 modell betöltve: {model_test.nbody} body, {model_test.nu} aktuátor')
del model_test


In [ ]:
# === 3. PROJEKT KÓD LETÖLTÉSE (GitHub) ===
import subprocess, os

REPO_URL = 'https://github.com/vorilevi/roboshelf-ai.git'
REPO_DIR = '/kaggle/working/roboshelf-ai'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(REPO_DIR)
print(f'📁 Munkamappa: {os.getcwd()}')
print(os.listdir('src/envs/'))
print(os.listdir('src/training/'))


In [ ]:
# === 4. G1 ÚTVONAL BEÁLLÍTÁSA A ENV-BEN ===
import os
os.environ['G1_MODEL_PATH'] = '/tmp/mujoco_menagerie/unitree_g1'
print(f'✅ G1_MODEL_PATH = {os.environ["G1_MODEL_PATH"]}')

In [ ]:
# === 5. TANÍTÁSI SZINT BEÁLLÍTÁSA ===
LEVEL = 'teljes'  # @param ['teszt', 'kozepes', 'teljes']

LEVELS = {
    'teszt': {
        'total_timesteps': 100_000,
        'n_steps': 2048, 'batch_size': 64, 'n_epochs': 10,
        'n_envs': 4, 'learning_rate': 3e-4, 'clip_range': 0.2,
        'description': 'Gyors teszt (~5 perc)',
    },
    'kozepes': {
        'total_timesteps': 5_000_000,
        'n_steps': 2048, 'batch_size': 64, 'n_epochs': 10,
        'n_envs': 4, 'learning_rate': 1e-4, 'clip_range': 0.15,
        'description': 'Közepes (~2 óra CPU-n)',
    },
    'teljes': {
        'total_timesteps': 10_000_000,
        'n_steps': 2048, 'batch_size': 64, 'n_epochs': 10,
        'n_envs': 4,   # 4 CPU core = 4 env optimális
        'learning_rate': 1e-4, 'clip_range': 0.15,
        'description': 'Teljes (~4 óra CPU-n, 10M lépés)',
    },
}

cfg = LEVELS[LEVEL]
print(f'📋 Szint: {LEVEL} — {cfg["description"]}')
print(f'   Timesteps: {cfg["total_timesteps"]:,}')
print(f'   Envs: {cfg["n_envs"]}× | Batch: {cfg["batch_size"]} | LR: {cfg["learning_rate"]}')
print(f'   Device: CPU (MlpPolicy GPU-n lassabb — SB3 issue #1245)')

In [ ]:
# === 6. RETAIL NAV ENV + PPO TANÍTÁS ===
import sys, os, time
import numpy as np
from pathlib import Path

sys.path.insert(0, 'src/envs')

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv, VecNormalize
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback, BaseCallback
from stable_baselines3.common.utils import set_random_seed
from roboshelf_retail_nav_env import RoboshelfRetailNavEnv
import torch

OUTPUT_DIR = Path('/kaggle/working/roboshelf-phase2-results')
MODELS_DIR = OUTPUT_DIR / 'models'
LOGS_DIR   = OUTPUT_DIR / 'logs'
BEST_DIR   = MODELS_DIR / 'best'
for d in [MODELS_DIR, LOGS_DIR, BEST_DIR, MODELS_DIR / 'checkpoints']:
    d.mkdir(parents=True, exist_ok=True)

timestamp = int(time.time())
run_name  = f'g1_retail_nav_{LEVEL}_{timestamp}'
device    = 'cpu'  # MlpPolicy CPU-n gyorsabb mint GPU-n (SB3 issue #1245)
print(f'🚀 {run_name}  |  device: {device}')

set_random_seed(42)
def make_single(rank, seed=42):
    def _init():
        env = RoboshelfRetailNavEnv()
        env.reset(seed=seed + rank)
        return env
    return _init

n_envs = cfg['n_envs']
# 'forkserver' stabilabb Kaggle-n mint 'fork': elkerüli a BrokenPipe hibát
# sok párhuzamos env indulásakor
if n_envs > 1:
    env = SubprocVecEnv([make_single(i) for i in range(n_envs)], start_method='forkserver')
else:
    env = DummyVecEnv([make_single(0)])
env = VecNormalize(env, norm_obs=True, norm_reward=True, clip_obs=10.0, clip_reward=10.0)
eval_env = DummyVecEnv([lambda: RoboshelfRetailNavEnv()])
eval_env = VecNormalize(eval_env, norm_obs=True, norm_reward=False, training=False)
print(f'✅ {n_envs}× env | obs={env.observation_space.shape} | act={env.action_space.shape}')

model = PPO(
    'MlpPolicy', env,
    policy_kwargs=dict(net_arch=dict(pi=[256, 256], vf=[256, 256])),
    learning_rate=cfg['learning_rate'],
    n_steps=cfg['n_steps'],
    batch_size=cfg['batch_size'],
    n_epochs=cfg['n_epochs'],
    gamma=0.99, gae_lambda=0.95,
    clip_range=cfg['clip_range'],
    ent_coef=0.001, vf_coef=0.5, max_grad_norm=0.5,
    tensorboard_log=str(LOGS_DIR),
    verbose=1, seed=42, device=device,
)

class SyncNormCb(BaseCallback):
    """VecNormalize stats szinkron — 10 lépésenként."""
    def __init__(self, train_env, eval_env):
        super().__init__()
        self.train_env, self.eval_env = train_env, eval_env
    def _on_step(self):
        if self.n_calls % 10 == 0:
            self.eval_env.obs_rms = self.train_env.obs_rms
        return True

class KaggleFlushCb(BaseCallback):
    """Kaggle IOPub timeout fix.
    
    Kaggle papermill 4 másodpercenként vár output-ot.
    GPU-n a tanítás gyors, de verbose=1 csak n_steps*n_envs-enként ír.
    Ez a callback rendszeresen flusholja és heartbeat sort ír.
    """
    def __init__(self, flush_every_n=200, heartbeat_every_n=2000):
        super().__init__()
        self.flush_every_n = flush_every_n
        self.heartbeat_every_n = heartbeat_every_n

    def _on_training_start(self):
        self._start_time = time.time()

    def _on_step(self):
        if self.n_calls % self.flush_every_n == 0:
            sys.stdout.flush()
        if self.n_calls % self.heartbeat_every_n == 0:
            elapsed = time.time() - self._start_time
            pct = 100 * self.num_timesteps / cfg['total_timesteps']
            print(f'  ⏳ {self.num_timesteps:,} lépés ({pct:.1f}%) | {elapsed/60:.1f} perc', flush=True)
        return True

eval_freq = max(cfg['total_timesteps'] // 20, 50_000)
chkpt_freq = max(cfg['total_timesteps'] // 10, 100_000)

eval_cb = EvalCallback(
    eval_env, best_model_save_path=str(BEST_DIR), log_path=str(LOGS_DIR),
    eval_freq=eval_freq, n_eval_episodes=5, deterministic=True,
)
chkpt_cb = CheckpointCallback(
    save_freq=chkpt_freq,
    save_path=str(MODELS_DIR / 'checkpoints'), name_prefix=run_name,
)

print(f'\n🚀 Tanítás indul: {cfg["total_timesteps"]:,} timestep...')
print(f'   Heartbeat: minden {2000 * n_envs:,} lépésnél')
start = time.time()
model.learn(
    total_timesteps=cfg['total_timesteps'],
    callback=[SyncNormCb(env, eval_env), eval_cb, chkpt_cb, KaggleFlushCb()],
    tb_log_name=run_name,
    progress_bar=False,  # progress_bar kikapcsolva: Kaggle-n néha IOPub problémát okoz
)
elapsed = time.time() - start
print(f'\n⏱️  Kész: {elapsed/60:.1f} perc ({elapsed/3600:.1f} óra)')

In [ ]:
# === 7. MENTÉS + KIÉRTÉKELÉS ===
final = str(MODELS_DIR / f'{run_name}_final')
model.save(f'{final}.zip')
env.save(f'{final}_vecnormalize.pkl')
env.save(str(BEST_DIR / 'best_model_vecnormalize.pkl'))
print(f'💾 Modell: {final}.zip')

ev = DummyVecEnv([lambda: RoboshelfRetailNavEnv()])
ev = VecNormalize.load(f'{final}_vecnormalize.pkl', ev)
ev.training = False
ev.norm_reward = False

rewards, lengths, dists = [], [], []
for ep in range(10):
    obs = ev.reset()
    done, total_r, steps = False, 0, 0
    while not done:
        act, _ = model.predict(obs, deterministic=True)
        obs, rew, done, info = ev.step(act)
        total_r += rew[0]; steps += 1
    rewards.append(total_r); lengths.append(steps)
    d = info[0].get('dist_to_target', float('nan'))
    dists.append(d)
    print(f'  Ep {ep+1:2d}: reward={total_r:7.1f}, lépés={steps:4d}, cél táv={d:.2f}m')

print(f'\n📈 Átlag reward: {np.mean(rewards):.1f} ± {np.std(rewards):.1f}')
print(f'📏 Átlag hossz:  {np.mean(lengths):.0f}')
start_dist = 3.3
progress = (start_dist - np.nanmean(dists)) / start_dist * 100
print(f'📍 Célhoz közeledés: {progress:.0f}% (start: {start_dist}m, végső: {np.nanmean(dists):.2f}m)')

In [ ]:
# === 8. LETÖLTÉS ===
import os
os.chdir('/kaggle/working')
!zip -r roboshelf_phase2_results.zip roboshelf-phase2-results/
print('📦 Letöltés: Kaggle Output fül → roboshelf_phase2_results.zip')
print()
print('MacBook-on kicsomagolás:')
print('  unzip ~/Downloads/roboshelf_phase2_results.zip -d ~/Documents/roboshelf-ai-dev/roboshelf-results/')